In [1]:
!pip install transformers

## Load Model

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import torch.nn.functional as F

MODEL_NAME = "Qwen/Qwen3.5-4B"

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16 if torch.cuda.is_available()
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto", trust_remote_code=True,
)
model.eval()

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2560)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=256

## Set Verbalizers

In [4]:
VERBALIZERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

OPTION_TOKEN_IDS = {}
for index, verbalizer in enumerate(VERBALIZERS):
    token_ids = tokenizer.encode(verbalizer, add_special_tokens=False)
    if len(token_ids) != 1:
        raise ValueError(f'Verbalizer "{verbalizer}" is not a single token: {token_ids}')
    OPTION_TOKEN_IDS[index] = token_ids[0]

## Building Prompt

In [5]:
def build_prompt(state, question, options, decision_rule="", extra_instruction=""):
    options_text = "\n".join(
        f"{VERBALIZERS[i]}. {option}" for i, option in enumerate(options)
    )
    return f"""STATE:

{state}

QUESTION:

{question}

DECISION RULE:

{decision_rule}

ADDITIONAL INSTRUCTIONS:

{extra_instruction}

OPTIONS:

{options_text}

Select the single option that is best supported by the STATE.

ANSWER:"""

## Forward Pass

In [6]:
@torch.no_grad()
def get_next_token_logits(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    return outputs.logits[0, -1, :]   # this is the next token after "ANSWER:"

## Logits to probabilities

In [7]:
def score_options(prompt, options):
    logits = get_next_token_logits(prompt)

    option_logits = {
        str(i): logits[OPTION_TOKEN_IDS[i]] for i in range(len(options))
    }
    labels = list(option_logits.keys())
    values = torch.stack([option_logits[label] for label in labels])

    # Normalize ONLY over the allowed options.
    probabilities_tensor = F.softmax(values, dim=0)

    probabilities = {label: float(probabilities_tensor[i]) for i, label in enumerate(labels)}
    raw_logits = {label: float(option_logits[label]) for label in labels}
    return probabilities, raw_logits

In [8]:
def calculate_winner(probabilities):
    ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)
    winner_index, winner_probability = int(ranked[0][0]), ranked[0][1]
    second_probability = ranked[1][1] if len(ranked) > 1 else 0.0
    return winner_index, winner_probability, second_probability, winner_probability - second_probability


def calculate_jev_score(probabilities):
    return sum(int(label) * p for label, p in probabilities.items())

## Wrapping into typed answer

In [9]:
def evaluate_field(state, question, options, field_type="choice", decision_rule="", extra_instruction=""):
    prompt = build_prompt(state, question, options, decision_rule, extra_instruction)
    probabilities, raw_logits = score_options(prompt, options)
    winner_index, winner_probability, second_probability, margin = calculate_winner(probabilities)

    result = {"type": field_type}
    if field_type == "score":
        result["score"] = calculate_jev_score(probabilities)
    elif field_type == "noul":
        result["choice"] = options[winner_index].lower()
    else:
        result["choice"] = options[winner_index]

    result["legend"] = {str(i): option for i, option in enumerate(options)}
    result["confidence"] = margin   # NOT calibrated confidence
    result["probabilities"] = probabilities
    result["stats"] = {}
    return result

## Example use case: State and Rubric

In [10]:
STATE = """
Mario is a software engineer who has worked professionally since 2020.

He built a React frontend and Node.js backend for a production SaaS
application, including APIs, PostgreSQL, and AWS infrastructure.

He led an architecture redesign of a real-time collaboration system
using WebSockets.

He mentored two junior engineers.
"""

In [14]:
# =========================================================
# RUBRIC
# =========================================================

RUBRIC = {

    # -----------------------------------------------------
    # TECHNICAL DEPTH
    # -----------------------------------------------------

    "technical_depth": {

        "type": "score",

        "question": """
What level of technical depth is explicitly demonstrated by this person?
""",

        "decision_rule": """
Evaluate the evidence using these dimensions:

1. Scope of ownership
2. Architectural responsibility
3. Technical breadth
4. Production experience
5. Technical complexity
6. Engineering leadership

Choose the highest level for which the STATE provides direct evidence.

Do not infer capabilities merely because a technology is mentioned.
""",

        "options": [
            (
                "No roles or projects where they wrote code. "
                "Technical exposure is adjacent only."
            ),

            (
                "Coding appears only as coursework, bootcamp, "
                "or tutorial projects."
            ),

            (
                "Small scoped work inside someone else's design: "
                "bug fixes, minor features, or CRUD screens."
            ),

            (
                "Owns features end to end in a live system and "
                "works across two layers."
            ),

            (
                "Owns whole systems and makes architecture tradeoffs. "
                "Deep in multiple technical domains."
            ),

            (
                "Deep specialist with exceptional breadth, systems "
                "expertise, or major architecture ownership."
            ),
        ],
    },


    # -----------------------------------------------------
    # MENTORSHIP
    # -----------------------------------------------------

    "mentorship_demonstrated": {

        "type": "noul",

        "question": """
Has this person demonstrated mentorship?
""",

        "decision_rule": """
Answer true only when the STATE explicitly provides evidence that
the person mentored, coached, supervised, or developed other people.
""",

        "options": [
            "false",
            "true",
        ],
    },



    # -----------------------------------------------------
    # PRIMARY TALENT PROFILE
    # -----------------------------------------------------

    "primary_talent_profile": {

        "type": "choice",

        "question": """
What is this person's primary talent profile?
""",

        "decision_rule": """
Choose the profile most strongly supported by the actual work
described in the STATE.

frontend_engineer = primarily frontend/UI work.

backend_engineer = primarily backend/API/server work.

full_stack_engineer = substantial frontend and backend work.

devops_engineer = primarily deployment, automation, CI/CD,
or infrastructure operations.

cloud_engineer = primarily cloud architecture or cloud platforms.

data_engineer = primarily data pipelines and data infrastructure.

ml_engineer = primarily machine learning systems.

mobile_engineer = primarily mobile applications.

engineering_manager = primarily people and engineering management.

architect = primarily system architecture and cross-system design.

Use explicit evidence rather than inferring from a single technology.
""",

        "options": [
            "frontend_engineer",
            "backend_engineer",
            "full_stack_engineer",
            "devops_engineer",
            "cloud_engineer",
            "data_engineer",
            "ml_engineer",
            "mobile_engineer",
            "engineering_manager",
            "architect",
        ],
    },
}

## Putting all together

In [16]:
answers = {}

for field_name, config in RUBRIC.items():

    answers[field_name] = evaluate_field(

        state=STATE,

        question=config["question"],

        options=config["options"],

        field_type=config["type"],

        decision_rule=config.get(
            "decision_rule",
            "",
        ),

        extra_instruction=config.get(
            "extra_instruction",
            "",
        ),
    )


# =========================================================
# FINAL JEV-STYLE OUTPUT
# =========================================================

result = {
    "model": MODEL_NAME,
    "answers": answers,
}


print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "model": "Qwen/Qwen3.5-4B",
  "answers": {
    "technical_depth": {
      "type": "score",
      "score": 2.5469970703125,
      "legend": {
        "0": "No roles or projects where they wrote code. Technical exposure is adjacent only.",
        "1": "Coding appears only as coursework, bootcamp, or tutorial projects.",
        "2": "Small scoped work inside someone else's design: bug fixes, minor features, or CRUD screens.",
        "3": "Owns features end to end in a live system and works across two layers.",
        "4": "Owns whole systems and makes architecture tradeoffs. Deep in multiple technical domains.",
        "5": "Deep specialist with exceptional breadth, systems expertise, or major architecture ownership."
      },
      "confidence": 0.3349609375,
      "probabilities": {
        "0": 0.10546875,
        "1": 0.1025390625,
        "2": 0.1279296875,
        "3": 0.48828125,
        "4": 0.1533203125,
        "5": 0.0220947265625
      },
      "stats": {}
    },
    